In [ ]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# If you haven't installed them then you use the colab directly. 
#   Choose Runtime -> Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have
# -> Python 3(ipykernel) -> Connect.
'''
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Read the Parquet file
file_path = r"/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet"
print(file_path)
# table = pq.read_table(file_path)
table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])
print(table.slice(0,5))

/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet


FileNotFoundError: /content/drive/MyDrive/AI/SampleFile/test-product-review.parquet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install if needed:
# pip install pandas pyarrow scikit-learn

import re
import pandas as pd
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Load the dataset
file_path = r"/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet"

table = pq.read_table(
    file_path,
    columns=["review_id", "review_body", "language", "stars"],
    filters=[[("language", "==", "en")]]
)

df = table.to_pandas()

print(df.head())
print(df.shape)
print(df["stars"].value_counts())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Data cleaning and label preparation

df = df[["review_id", "review_body", "stars"]].dropna().copy()

# Convert stars to numeric
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep only clear sentiment examples
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# Remove neutral examples for a simpler binary sentiment task
df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df["sentiment"].value_counts())

In [ ]:
# 3) Text preprocessing

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review_body"].apply(clean_text)

# Optional: drop empty reviews after cleaning
df = df[df["clean_review"] != ""].reset_index(drop=True)

print(df[["review_body", "clean_review", "sentiment"]].head())

In [ ]:
# 4) Train/Test split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Class distribution in train:", y_train.value_counts().to_dict())